# Step 15 — replace parent blocks with selected child blocks

**# of cells in notebook:** 1

**Purpose:** Create the final citywide blocks layer by replacing each selected parent block with its final child-block subdivision. Multipart-resolved outputs from Step 14 take priority where they exist; otherwise the canonical selected child blocks are used. This step also creates citywide binary lineage fields identifying whether each final block feature resulted from splitting and whether BEAM was the splitting method.

**Input:**

- the master citywide blocks feature class
- `heterogeneous_largePop_selection`
- `heterogeneous_largePop_blocks_MPexplode`
- `beam_selection_summary.csv` from Step 12
- within each selected block folder:
  - `new_blocks_populated.gpkg`
- child fields:
  - `block_id`
  - `population`

**Output:**

- `johannesburg_blocks_utm35s_hetero_swapped` — a copy of the master citywide blocks feature class in which selected parent blocks have been replaced by their child blocks
- `was_split` and `beam` binary fields in the final citywide dataset, with `was_split` immediately before `beam` in the attribute list:
  - unsplit original block: `was_split = 0`, `beam = 0`
  - standard split child: `was_split = 1`, `beam = 0`
  - BEAM split child: `was_split = 1`, `beam = 1`
- a CSV summarizing processed, skipped, and failed parent-block replacements

**Main logic:**

**Cell 1 — Swap selected child blocks into the master blocks layer**

1. Creates a copy of the master citywide blocks feature class.
2. Adds `was_split` followed by `beam` to the copied citywide schema and initializes both to 0 for the original unsplit features.
3. Reads `beam_selection_summary.csv` and identifies parent blocks whose canonical selection was replaced by a BEAM result.
4. Builds an effective list of source-block folders, giving Step 14 MPexplode outputs priority wherever they exist and otherwise using the canonical selected outputs.
5. Reads the original parent block's editable non-system attributes.
6. Reads each selected child feature's geometry, final `block_id`, and recalculated `population`.
7. Projects child geometry to the master feature class CRS when necessary.
8. Inserts each child with `was_split = 1`; sets `beam = 1` when its parent is identified as a selected BEAM parent and otherwise sets `beam = 0`.
9. Assigns each child its own `block_id` and `population` while inheriting the remaining editable parent attributes.
10. Deletes the original parent feature only after the child features have been inserted successfully.
11. Writes a detailed swap summary CSV.


In [ ]:
"""
Swap selected heterogeneous block subdivisions into a copy of the master blocks
feature class, using two possible selection roots. The final citywide output
adds was_split followed by beam as binary lineage fields. BEAM status is read
from the Step 12 beam_selection_summary.csv rather than from child attributes.

Priority rule:
1. If a block folder exists in heterogeneous_largePop_blocks_MPexplode, use that.
2. Otherwise, use the block folder from heterogeneous_largePop_selection.

This allows multipart-exploded and repopulated outputs to override the original
selection outputs only where they exist.

Run in the ArcGIS Pro Python environment.
"""

import csv
import os
import re
import sys
import traceback

import arcpy


# ---------------------------------------------------------------------
# Inputs
# ---------------------------------------------------------------------

master_fc = (
    r"E:\_johannesburg\_analysis\blocks\blocks.gdb"
    r"\johannesburg_blocks_utm35s"
)

selection_root_original = (
    r"E:\_johannesburg\_analysis"
    r"\heterogeneous_largePop_selection"
)

selection_root_mpexplode = (
    r"E:\_johannesburg\_analysis"
    r"\heterogeneous_largePop_blocks_MPexplode"
)

gpkg_name = "new_blocks_populated.gpkg"

beam_selection_summary_csv = os.path.join(
    selection_root_original,
    "beam_selection_summary.csv",
)

out_fc = (
    r"E:\_johannesburg\_analysis\blocks\blocks.gdb"
    r"\johannesburg_blocks_utm35s_hetero_swapped"
)

summary_csv = (
    r"E:\_johannesburg\_analysis\blocks"
    r"\johannesburg_heterogeneous_largePop_selection_swap_summary.csv"
)

# Set to True only if you intentionally want to recreate out_fc.
overwrite_output_copy = False

# ---------------------------------------------------------------------
# Field settings
# ---------------------------------------------------------------------

child_block_id_field = "block_id"
child_population_field = "population"

master_block_id_field = "block_id"
master_population_field = "population"
master_was_split_field = "was_split"
master_beam_field = "beam"

# If None, the script automatically inherits all editable parent fields
# except block_id and population.
#
# If you want to inherit only specific fields, replace None with a list, e.g.:
# inherited_parent_fields = ["HH_CC", "HH_Grtr10ha", "LargePop"]
inherited_parent_fields = None

# If a GeoPackage has more than one polygon layer, safest default is to skip it.
# Options: "skip" or "first"
multiple_layer_mode = "skip"


# ---------------------------------------------------------------------
# Helper functions
# ---------------------------------------------------------------------

def die(message):
    print(f"ERROR: {message}")
    sys.exit(1)


def field_names(dataset):
    return [f.name for f in arcpy.ListFields(dataset)]


def validate_fields(dataset, required_fields, dataset_label):
    existing = set(field_names(dataset))
    missing = [f for f in required_fields if f not in existing]
    if missing:
        die(
            f"{dataset_label} is missing required fields: {missing}\n"
            f"Dataset: {dataset}"
        )


def sql_eq_text(dataset, field_name, value):
    fld = arcpy.AddFieldDelimiters(dataset, field_name)
    escaped = str(value).replace("'", "''")
    return f"{fld} = '{escaped}'"


def parse_parent_block_id(folder_name):
    """
    Convert folder name like '_136' to master block_id 'blk_136'.
    """
    m = re.fullmatch(r"_(\d+)", folder_name)
    if not m:
        return None, None

    source_block = int(m.group(1))
    parent_block_id = f"blk_{source_block}"
    return source_block, parent_block_id


def normalize_source_block_value(value):
    """Normalize values such as blk_52730, _52730, 52730, or 52730.0."""
    if value is None:
        return None

    text = str(value).strip()
    if not text:
        return None

    if re.fullmatch(r"\d+\.0", text):
        return text[:-2]
    if re.fullmatch(r"\d+", text):
        return text

    matches = re.findall(r"\d+", text)
    if not matches:
        return None

    return matches[-1]


def load_selected_beam_source_blocks():
    """
    Read Step 12 beam_selection_summary.csv and return numeric parent IDs for
    rows whose status is 'selected'.
    """
    if not os.path.exists(beam_selection_summary_csv):
        print(
            "WARNING: BEAM selection summary not found. "
            "Assuming no final child blocks are BEAM-derived."
        )
        print(f"  {beam_selection_summary_csv}")
        return set()

    selected = set()

    with open(beam_selection_summary_csv, "r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)

        fieldnames = reader.fieldnames or []
        required = {"status", "lookup_block_id"}
        missing = sorted(required.difference(fieldnames))
        if missing:
            die(
                "BEAM selection summary is missing required field(s): "
                + ", ".join(missing)
            )

        for row in reader:
            if str(row.get("status", "")).strip().lower() != "selected":
                continue

            source_id = normalize_source_block_value(
                row.get("lookup_block_id")
            )
            if source_id is not None:
                selected.add(source_id)

    print(f"Selected BEAM parent blocks loaded: {len(selected)}")
    return selected


def block_folder_names(root):
    if not os.path.isdir(root):
        return set()

    return {
        d for d in os.listdir(root)
        if os.path.isdir(os.path.join(root, d)) and re.fullmatch(r"_\d+", d)
    }


def build_effective_folder_list():
    """
    Build block-folder source list using MPexplode as the override root.
    """
    original_folders = block_folder_names(selection_root_original)
    mpexplode_folders = block_folder_names(selection_root_mpexplode)

    all_folders = sorted(
        original_folders.union(mpexplode_folders),
        key=lambda x: int(x[1:])
    )

    records = []

    for folder in all_folders:
        original_folder_path = os.path.join(selection_root_original, folder)
        mpexplode_folder_path = os.path.join(selection_root_mpexplode, folder)

        original_gpkg = os.path.join(original_folder_path, gpkg_name)
        mpexplode_gpkg = os.path.join(mpexplode_folder_path, gpkg_name)

        if folder in mpexplode_folders:
            selected_root = selection_root_mpexplode
            selected_folder_path = mpexplode_folder_path
            selected_gpkg = mpexplode_gpkg
            source_used = "MPexplode"
        else:
            selected_root = selection_root_original
            selected_folder_path = original_folder_path
            selected_gpkg = original_gpkg
            source_used = "original_selection"

        records.append({
            "folder": folder,
            "selected_root": selected_root,
            "selected_folder_path": selected_folder_path,
            "selected_gpkg": selected_gpkg,
            "source_used": source_used,
            "exists_in_original": folder in original_folders,
            "exists_in_mpexplode": folder in mpexplode_folders,
        })

    return records


def list_polygon_layers_in_gpkg(gpkg_path):
    """
    Return full paths to polygon feature classes inside a GeoPackage.
    """
    layers = []

    old_workspace = arcpy.env.workspace

    try:
        arcpy.env.workspace = gpkg_path
        fcs = arcpy.ListFeatureClasses() or []

        for fc in fcs:
            fc_path = os.path.join(gpkg_path, fc)

            try:
                desc = arcpy.Describe(fc_path)
                if desc.shapeType.lower() == "polygon":
                    layers.append(fc_path)
            except Exception:
                pass

    finally:
        arcpy.env.workspace = old_workspace

    # Fallback for cases where ListFeatureClasses is incomplete.
    if not layers:
        try:
            for dirpath, _dirnames, filenames in arcpy.da.Walk(
                gpkg_path,
                datatype="FeatureClass",
                type="Polygon",
            ):
                for name in filenames:
                    layers.append(os.path.join(dirpath, name))
        except Exception:
            pass

    return sorted(set(layers))


def auto_inherited_fields(master_fc_path):
    """
    Automatically inherit editable non-system parent fields.

    Excludes:
      - geometry/OID/system fields
      - block_id, because child block_id replaces parent block_id
      - population, because child population replaces parent population
    """
    exclude = {
        master_block_id_field.lower(),
        master_population_field.lower(),
        master_was_split_field.lower(),
        master_beam_field.lower(),
    }

    out = []

    for f in arcpy.ListFields(master_fc_path):
        if f.name.lower() in exclude:
            continue

        if f.type in {
            "OID",
            "Geometry",
            "Blob",
            "Raster",
            "GUID",
            "GlobalID",
        }:
            continue

        if not f.editable:
            continue

        out.append(f.name)

    return out


def resolve_inherited_fields(master_fc_path):
    if inherited_parent_fields is None:
        fields = auto_inherited_fields(master_fc_path)
        print("\nAuto-detected inherited parent fields:")
        for f in fields:
            print(f"  {f}")
        return fields

    existing = set(field_names(master_fc_path))
    keep = []
    missing = []

    for f in inherited_parent_fields:
        if f in existing:
            keep.append(f)
        else:
            missing.append(f)

    if missing:
        print("\nWARNING: Some requested inherited fields do not exist in master_fc:")
        for f in missing:
            print(f"  {f}")
        print("They will be ignored.")

    return keep


def get_one_parent_row(out_fc_path, parent_block_id, parent_fields):
    """
    Read the parent row inherited values from the copied master FC.
    """
    where = sql_eq_text(out_fc_path, master_block_id_field, parent_block_id)
    read_fields = ["OID@", master_block_id_field] + parent_fields

    matches = []

    with arcpy.da.SearchCursor(
        out_fc_path,
        read_fields,
        where_clause=where,
    ) as cur:
        for row in cur:
            matches.append(row)

    if len(matches) == 0:
        return None, f"Parent block not found in output copy: {parent_block_id}"

    if len(matches) > 1:
        return None, f"More than one parent block found in output copy: {parent_block_id}"

    row = matches[0]
    inherited_values = dict(zip(parent_fields, row[2:]))

    return inherited_values, None


def read_child_rows(layer_path, target_sr, beam_flag):
    """
    Read child geometry, block_id, and population from the selected GeoPackage.

    Every selected child is a split feature, so was_split is always 1.
    beam_flag is determined once from the Step 12 BEAM-selection summary.
    """
    validate_fields(
        layer_path,
        [
            child_block_id_field,
            child_population_field,
        ],
        f"Child layer {layer_path}",
    )

    rows = []

    with arcpy.da.SearchCursor(
        layer_path,
        [
            "SHAPE@",
            child_block_id_field,
            child_population_field,
        ],
    ) as cur:
        for geom, child_bid, child_pop in cur:
            if geom is None:
                continue

            if child_bid in (None, ""):
                raise ValueError(
                    f"Child feature has empty block_id in {layer_path}"
                )

            try:
                child_sr = geom.spatialReference

                if (
                    target_sr is not None
                    and child_sr is not None
                    and child_sr.factoryCode not in (None, 0)
                    and target_sr.factoryCode not in (None, 0)
                    and child_sr.factoryCode != target_sr.factoryCode
                ):
                    geom = geom.projectAs(target_sr)

            except Exception:
                pass

            rows.append(
                (
                    geom,
                    child_bid,
                    child_pop,
                    1,
                    int(beam_flag),
                )
            )

    return rows


def insert_child_rows(out_fc_path, child_rows, inherited_values, parent_fields):
    insert_fields = [
        "SHAPE@",
        master_block_id_field,
        master_population_field,
        master_was_split_field,
        master_beam_field,
    ] + parent_fields

    inserted = 0

    with arcpy.da.InsertCursor(out_fc_path, insert_fields) as icur:
        for geom, child_bid, child_pop, child_was_split, child_beam in child_rows:
            values = [
                geom,
                child_bid,
                child_pop,
                child_was_split,
                child_beam,
            ]
            values.extend(inherited_values.get(f) for f in parent_fields)

            icur.insertRow(values)
            inserted += 1

    return inserted


def delete_parent_row(out_fc_path, parent_block_id):
    where = sql_eq_text(out_fc_path, master_block_id_field, parent_block_id)

    deleted = 0

    with arcpy.da.UpdateCursor(
        out_fc_path,
        [master_block_id_field],
        where_clause=where,
    ) as ucur:
        for _row in ucur:
            ucur.deleteRow()
            deleted += 1

    return deleted


def ensure_lineage_fields(out_fc_path):
    """
    Ensure was_split immediately precedes beam in the output schema, then
    initialize both to 0 for the copied unsplit citywide features.
    """
    existing = field_names(out_fc_path)
    lower = [f.lower() for f in existing]

    has_was_split = master_was_split_field.lower() in lower
    has_beam = master_beam_field.lower() in lower

    if not has_was_split and not has_beam:
        arcpy.management.AddField(
            out_fc_path,
            master_was_split_field,
            "SHORT",
        )
        arcpy.management.AddField(
            out_fc_path,
            master_beam_field,
            "SHORT",
        )
    elif has_was_split and has_beam:
        existing = field_names(out_fc_path)
        was_idx = existing.index(
            next(f for f in existing if f.lower() == master_was_split_field.lower())
        )
        beam_idx = existing.index(
            next(f for f in existing if f.lower() == master_beam_field.lower())
        )
        if was_idx > beam_idx:
            die(
                f"Existing field order places {master_beam_field} before "
                f"{master_was_split_field}. Recreate the output from a master "
                "without those fields so was_split can precede beam."
            )
    else:
        die(
            "Only one lineage field already exists in the output copy. "
            "Recreate the output from a master without was_split/beam so the "
            "required field order can be guaranteed."
        )

    with arcpy.da.UpdateCursor(
        out_fc_path,
        [master_was_split_field, master_beam_field],
    ) as ucur:
        for row in ucur:
            row[0] = 0
            row[1] = 0
            ucur.updateRow(row)


# ---------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------

def main():
    arcpy.env.overwriteOutput = overwrite_output_copy

    if not arcpy.Exists(master_fc):
        die(f"Master feature class does not exist: {master_fc}")

    if not os.path.isdir(selection_root_original):
        die(f"Original selection root does not exist: {selection_root_original}")

    if not os.path.isdir(selection_root_mpexplode):
        print(
            "WARNING: MPexplode selection root does not exist. "
            "Only original selection root will be used."
        )
        print(f"  {selection_root_mpexplode}")

    validate_fields(
        master_fc,
        [master_block_id_field, master_population_field],
        "Master feature class",
    )

    parent_fields = resolve_inherited_fields(master_fc)

    validate_fields(
        master_fc,
        [master_block_id_field, master_population_field] + parent_fields,
        "Master feature class",
    )

    if arcpy.Exists(out_fc):
        if overwrite_output_copy:
            print(f"Deleting existing output copy: {out_fc}")
            arcpy.management.Delete(out_fc)
        else:
            die(
                "Output copy already exists. To avoid accidental overwrite, "
                "either delete it, choose a new out_fc name, or set "
                "overwrite_output_copy = True.\n"
                f"Existing output: {out_fc}"
            )

    print("\nCopying master feature class...")
    print(f"  From: {master_fc}")
    print(f"  To:   {out_fc}")
    arcpy.management.CopyFeatures(master_fc, out_fc)

    # Add lineage fields in the required attribute order and initialize the
    # copied unsplit citywide blocks to 0/0.
    ensure_lineage_fields(out_fc)

    target_sr = arcpy.Describe(out_fc).spatialReference

    selected_beam_source_blocks = load_selected_beam_source_blocks()

    folder_records = build_effective_folder_list()

    print(f"\nEffective block folders found: {len(folder_records)}")
    print("Priority rule: MPexplode overrides original selection where present.")

    n_mpexplode = sum(1 for r in folder_records if r["source_used"] == "MPexplode")
    n_original = sum(1 for r in folder_records if r["source_used"] == "original_selection")

    print(f"  Using MPexplode folders:          {n_mpexplode}")
    print(f"  Using original selection folders: {n_original}")

    summary_rows = []

    total_inserted = 0
    total_deleted = 0
    processed = 0
    skipped = 0
    errored = 0

    for rec in folder_records:
        folder = rec["folder"]
        source_used = rec["source_used"]
        gpkg_path = rec["selected_gpkg"]

        source_block, parent_block_id = parse_parent_block_id(folder)
        beam_flag = int(str(source_block) in selected_beam_source_blocks)

        status = ""
        message = ""
        layer_used = ""
        child_count = 0
        inserted_count = 0
        deleted_count = 0

        print("\n" + "=" * 72)
        print(f"Processing {folder} -> parent {parent_block_id}")
        print(f"  Source used: {source_used}")
        print(f"  GPKG:        {gpkg_path}")
        print(f"  was_split:   1")
        print(f"  beam:        {beam_flag}")

        try:
            if not os.path.exists(gpkg_path):
                status = "skipped"
                message = f"No {gpkg_name} found in selected source"
                print(f"  Skipping: {message}")
                skipped += 1

            else:
                layers = list_polygon_layers_in_gpkg(gpkg_path)

                if len(layers) == 0:
                    status = "skipped"
                    message = "GeoPackage contains no polygon feature layers"
                    print(f"  Skipping: {message}")
                    skipped += 1

                elif len(layers) > 1 and multiple_layer_mode == "skip":
                    status = "skipped"
                    message = (
                        "GeoPackage contains multiple polygon layers; "
                        f"skipped for safety: {layers}"
                    )
                    print(f"  Skipping: {message}")
                    skipped += 1

                else:
                    layer_path = layers[0]
                    layer_used = os.path.basename(layer_path)

                    print(f"  Layer: {layer_path}")

                    inherited_values, err = get_one_parent_row(
                        out_fc,
                        parent_block_id,
                        parent_fields,
                    )

                    if err:
                        status = "skipped"
                        message = err
                        print(f"  Skipping: {message}")
                        skipped += 1

                    else:
                        child_rows = read_child_rows(
                            layer_path,
                            target_sr,
                            beam_flag,
                        )
                        child_count = len(child_rows)

                        print(f"  Child rows read: {child_count}")

                        if child_count == 0:
                            status = "skipped"
                            message = "Layer has zero readable child features"
                            print(f"  Skipping: {message}")
                            skipped += 1

                        else:
                            # Insert children first.
                            # Delete parent only after insert succeeds.
                            inserted_count = insert_child_rows(
                                out_fc,
                                child_rows,
                                inherited_values,
                                parent_fields,
                            )

                            deleted_count = delete_parent_row(
                                out_fc,
                                parent_block_id,
                            )

                            status = "processed"
                            message = (
                                "Inserted child rows and deleted parent row"
                            )

                            print(f"  Inserted children:   {inserted_count}")
                            print(f"  Deleted parent rows: {deleted_count}")

                            processed += 1
                            total_inserted += inserted_count
                            total_deleted += deleted_count

            summary_rows.append({
                "folder": folder,
                "source_block": source_block,
                "parent_block_id": parent_block_id,
                "source_used": source_used,
                "was_split": 1,
                "beam": beam_flag,
                "exists_in_original": rec["exists_in_original"],
                "exists_in_mpexplode": rec["exists_in_mpexplode"],
                "gpkg_path": gpkg_path,
                "layer_used": layer_used,
                "child_count": child_count,
                "inserted_count": inserted_count,
                "deleted_parent_count": deleted_count,
                "status": status,
                "message": message,
            })

        except Exception as ex:
            errored += 1

            msg = f"{type(ex).__name__}: {ex}"
            print(f"  ERROR: {msg}")
            traceback.print_exc()

            summary_rows.append({
                "folder": folder,
                "source_block": source_block,
                "parent_block_id": parent_block_id,
                "source_used": source_used,
                "was_split": 1,
                "beam": beam_flag,
                "exists_in_original": rec["exists_in_original"],
                "exists_in_mpexplode": rec["exists_in_mpexplode"],
                "gpkg_path": gpkg_path,
                "layer_used": layer_used,
                "child_count": child_count,
                "inserted_count": inserted_count,
                "deleted_parent_count": deleted_count,
                "status": "error",
                "message": msg,
            })

    # Write summary CSV.
    os.makedirs(os.path.dirname(summary_csv), exist_ok=True)

    with open(summary_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "folder",
                "source_block",
                "parent_block_id",
                "source_used",
                "was_split",
                "beam",
                "exists_in_original",
                "exists_in_mpexplode",
                "gpkg_path",
                "layer_used",
                "child_count",
                "inserted_count",
                "deleted_parent_count",
                "status",
                "message",
            ],
        )

        writer.writeheader()
        writer.writerows(summary_rows)

    print("\n" + "=" * 72)
    print("Done.")
    print(f"Output feature class: {out_fc}")
    print(f"Summary CSV:          {summary_csv}")
    print(f"Processed folders:    {processed}")
    print(f"Skipped folders:      {skipped}")
    print(f"Errored folders:      {errored}")
    print(f"Total child inserted: {total_inserted}")
    print(f"Total parent deleted: {total_deleted}")
    print(f"Net row change:       {total_inserted - total_deleted}")

    print("\nSource selection:")
    print(f"  MPexplode used:          {n_mpexplode}")
    print(f"  Original selection used: {n_original}")


if __name__ == "__main__":
    main()
